# GR × pearson — transparent Test regression AD

This is a self-contained, cell-by-cell inference and applicability-domain notebook. It does not clone GitHub and does not import the project AD runtime. Every computational stage is visible, prints a compact audit, and saves a reloadable artifact. Frozen inputs are loaded only; nothing is fitted or trained.

## Data lineage

Raw receptor CSV → locked Train/Validation/Test → four fresh 256-D feature families → HStack1024 → frozen MinMaxScaler → frozen selector → frozen regressor + kNN AD.

`No AD` uses every Test row. Each `k=` row reports regression metrics only for Test rows classified as IND. The Case-study dataset is not used here.

In [ ]:
# STEP 0 — Configuration. Edit paths here when auto-discovery is ambiguous.
RUN_DATASET = 'GR'
RUN_METHOD = 'pearson'
EXPECTED_SELECTED_K = 30
EXPECTED_MODEL_CLASS = 'Ridge'
EXPECTED_REFERENCE_TEST_R2 = 0.519489428395004
EXPECTED_N_VAL = 327
EXPECTED_N_TEST = 328
RAW_CSV_FILENAME = 'GR_regression.csv'
FROZEN_BUNDLE_FILENAME = 'GR_pearson_best_model.pkl'

# AD controls. One summary/detail CSV is written per multiplier.
AD_THRESHOLD_MULTIPLIERS = (0.5, 2.0)
AD_NEIGHBOR_K_VALUES = tuple(range(3, 26))
R2_TOLERANCE = 0.001

# Leave a path blank to auto-discover the exact filename under INPUT_SEARCH_ROOTS.
# If Kaggle finds multiple copies, paste the desired full path here.
RAW_CSV_PATH = ""
SMILES_CHECKPOINT_PATH = ""
SELFIES_CHECKPOINT_PATH = ""
GRAPH_CHECKPOINT_PATH = ""
FINGERPRINT_CHECKPOINT_PATH = ""
FINGERPRINT_TRANSFORMER_PATH = ""

# Preferred: one monolithic *.pkl bundle or a component-bundle *.json descriptor.
FROZEN_BUNDLE_PATH = ""
# Alternative: leave FROZEN_BUNDLE_PATH blank and provide all three paths below.
FROZEN_MODEL_PATH = ""
FROZEN_SCALER_PATH = ""
FROZEN_SELECTOR_PATH = ""

# Safe fallback for the small variant-specific final artifacts. External paths
# take precedence; set False to require an attached Kaggle dataset instead.
USE_EMBEDDED_FROZEN_ARTIFACTS = True

INPUT_SEARCH_ROOTS = ["/kaggle/input"]
OUTPUT_ROOT = f"/kaggle/working/AD_Test_Regression_{RUN_DATASET}_{RUN_METHOD}"

# Kaggle dependency setup. Replace SKFP_INSTALL_SPEC with an attached wheel path
# if you do not want pip to fetch the pinned third-party fingerprint package.
AUTO_INSTALL_MISSING_DEPENDENCIES = True
SKFP_INSTALL_SPEC = "git+https://github.com/plenoi/scikit-finger-plenoi.git@master"

EXPECTED_CANONICAL_HASHES = {}
EXPECTED_CANONICAL_SMILES_HASHES = {'train': '4b69da7f4fac25e9d251f38136022e44143975a0c20c6652e77e0498fb076492', 'val': '0af3bab48d9bd5865eabc3a8fb9dc20d0412b9f0b6beb4e616ef7d88e1d6f251', 'test': 'e73fd2bd0b5fc60a27380396b42370c73b3083ac0ac54329b077c04b38e845f9'}
EXPECTED_CANONICAL_PAIRED_HASHES = {'train': 'c7e78e1e6ba4e1a4be1a3411c5f07351d415c62d73d708b9dc9dc7a5fc462cf4', 'val': 'd38eb41bdcfffd6b49e6002e85f21ab51294eeced8fe089416de6037cab68435', 'test': '5f4afef9c3e9270728a4543b52776b7ae901dd7db549bb7f7f128c662f623be0'}

assert RUN_DATASET in {"AR", "ER", "GR", "PR"}
assert RUN_METHOD in {"mutual_info", "pearson"}
assert AD_THRESHOLD_MULTIPLIERS
assert tuple(AD_NEIGHBOR_K_VALUES) == tuple(range(3, 26))
print("Variant:", f"{RUN_DATASET} × {RUN_METHOD}")
print("Expected frozen model:", EXPECTED_MODEL_CLASS)
print("Expected selected features:", EXPECTED_SELECTED_K)
print("Output root:", OUTPUT_ROOT)

In [ ]:
# STEP 0B — Install only missing third-party inference dependencies.
import importlib.util
import os
import subprocess
import sys

os.environ.setdefault("WANDB_MODE", "disabled")
os.environ.setdefault("WANDB_SILENT", "true")

required_modules = {
    "transformers": "transformers==4.48.3",
    "sentencepiece": "sentencepiece",
    "selfies": "selfies",
    "rdkit": "rdkit",
    "deepchem": "deepchem",
    "torch_geometric": "torch-geometric",
    "skfp": SKFP_INSTALL_SPEC,
}
missing_specs = [
    package_spec
    for module_name, package_spec in required_modules.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_specs:
    if not AUTO_INSTALL_MISSING_DEPENDENCIES:
        raise ModuleNotFoundError(
            "Missing dependencies: " + ", ".join(missing_specs)
            + ". Install them or enable AUTO_INSTALL_MISSING_DEPENDENCIES."
        )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_specs])

from transformers import AutoModel, AutoTokenizer

def pin_remote_code_revision(auto_class):
    original = auto_class.from_pretrained
    if getattr(original, "_hstack_revision_pinned", False):
        return
    def pinned(pretrained_model_name_or_path, *args, **kwargs):
        revision = kwargs.get("revision")
        if revision:
            kwargs.setdefault("code_revision", revision)
        return original(pretrained_model_name_or_path, *args, **kwargs)
    pinned._hstack_revision_pinned = True
    auto_class.from_pretrained = pinned

pin_remote_code_revision(AutoTokenizer)
pin_remote_code_revision(AutoModel)
print("Third-party dependencies: OK")
print("Inference device contract: CPU")

In [ ]:
# STEP 0C — Materialize the small frozen final artifacts when no Kaggle input contains them.
# These payloads are binary sklearn/joblib artifacts, not executable project source.
import base64
import hashlib
from pathlib import Path

EMBEDDED_FROZEN_FILES_B64 = {'GR_pearson_best_model.pkl': '...'}
EMBEDDED_FROZEN_SHA256 = {'GR_pearson_best_model.pkl': 'b5a6ae6f3ecaf1cdcd8ff129ebc4d0da8c1064a2b3f9e6b0897cf72593cac7e8'}
EMBEDDED_FROZEN_ROOT = (
    Path(OUTPUT_ROOT).expanduser().parent
    / "ad_embedded_frozen_inputs"
    / f"{RUN_DATASET}_{RUN_METHOD}"
)

def materialize_embedded_frozen_files():
    EMBEDDED_FROZEN_ROOT.mkdir(parents=True, exist_ok=True)
    resolved = {}
    for filename, encoded in EMBEDDED_FROZEN_FILES_B64.items():
        data = base64.b64decode(encoded.encode("ascii"))
        actual = hashlib.sha256(data).hexdigest()
        expected = EMBEDDED_FROZEN_SHA256[filename]
        if actual != expected:
            raise RuntimeError(f"Embedded frozen artifact checksum failed: {filename}")
        path = EMBEDDED_FROZEN_ROOT / filename
        if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            path.write_bytes(data)
        resolved[filename] = path.resolve()
    return resolved

EMBEDDED_FROZEN_PATHS = (
    materialize_embedded_frozen_files()
    if USE_EMBEDDED_FROZEN_ARTIFACTS
    else {}
)
print("Embedded frozen fallback:", "ready" if EMBEDDED_FROZEN_PATHS else "disabled")
print("Embedded frozen SHA-256:", EMBEDDED_FROZEN_SHA256)

In [ ]:
# STEP 1 — Imports, output directories, and transparent input-path resolution.
import ast
import hashlib
import json
import random
import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

OUTPUT_DIR = Path(OUTPUT_ROOT) / RUN_DATASET / RUN_METHOD
ARTIFACT_DIR = OUTPUT_DIR / "intermediate_artifacts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def _configured_path(value):
    return Path(value).expanduser().resolve() if str(value).strip() else None

def discover_exact_file(label, configured, filenames, *, required=True):
    explicit = _configured_path(configured)
    if explicit is not None:
        if not explicit.is_file():
            raise FileNotFoundError(f"{label} path does not exist: {explicit}")
        return explicit
    matches = []
    for root_value in INPUT_SEARCH_ROOTS:
        root = Path(root_value).expanduser()
        if not root.is_dir():
            continue
        for filename in filenames:
            matches.extend(path.resolve() for path in root.rglob(filename) if path.is_file())
    matches = sorted(set(matches))
    if len(matches) == 1:
        return matches[0]
    if not matches and not required:
        return None
    if not matches:
        raise FileNotFoundError(
            f"Could not find {label}. Set its *_PATH value in STEP 0. "
            f"Expected one of: {list(filenames)}"
        )
    preview = "\n".join(f"  - {path}" for path in matches[:20])
    raise RuntimeError(
        f"Found multiple candidates for {label}; set the exact path in STEP 0:\n{preview}"
    )

INPUT_PATHS = {
    "raw_csv": discover_exact_file("raw CSV", RAW_CSV_PATH, [RAW_CSV_FILENAME]),
    "smiles_checkpoint": discover_exact_file(
        "SMILES checkpoint", SMILES_CHECKPOINT_PATH, [f"model_smiles_{RUN_DATASET}.pt"]
    ),
    "selfies_checkpoint": discover_exact_file(
        "SELFIES checkpoint", SELFIES_CHECKPOINT_PATH, [f"model_selfies_{RUN_DATASET}.pt"]
    ),
    "graph_checkpoint": discover_exact_file(
        "Graph checkpoint", GRAPH_CHECKPOINT_PATH, [f"model_graph_{RUN_DATASET}.pt"]
    ),
    "fingerprint_checkpoint": discover_exact_file(
        "Fingerprint checkpoint", FINGERPRINT_CHECKPOINT_PATH,
        [f"model_fingerprint_{RUN_DATASET}.pt"],
    ),
    "fingerprint_transformer": discover_exact_file(
        "Fingerprint transformer", FINGERPRINT_TRANSFORMER_PATH,
        [f"ecfp_transformer_{RUN_DATASET}.pkl"],
    ),
}

separate_frozen_paths = all(
    str(value).strip()
    for value in (FROZEN_MODEL_PATH, FROZEN_SCALER_PATH, FROZEN_SELECTOR_PATH)
)
if separate_frozen_paths:
    INPUT_PATHS["frozen_model"] = discover_exact_file(
        "frozen model", FROZEN_MODEL_PATH, [Path(FROZEN_MODEL_PATH).name]
    )
    INPUT_PATHS["frozen_scaler"] = discover_exact_file(
        "frozen scaler", FROZEN_SCALER_PATH, [Path(FROZEN_SCALER_PATH).name]
    )
    INPUT_PATHS["frozen_selector"] = discover_exact_file(
        "frozen selector", FROZEN_SELECTOR_PATH, [Path(FROZEN_SELECTOR_PATH).name]
    )
else:
    if any(str(value).strip() for value in (FROZEN_MODEL_PATH, FROZEN_SCALER_PATH, FROZEN_SELECTOR_PATH)):
        raise ValueError("Provide all three separate frozen component paths, or leave all three blank")
    discovered_bundle = discover_exact_file(
        "frozen bundle", FROZEN_BUNDLE_PATH, [FROZEN_BUNDLE_FILENAME], required=False
    )
    if discovered_bundle is not None:
        INPUT_PATHS["frozen_bundle"] = discovered_bundle
        FROZEN_INPUT_SOURCE = "external Kaggle input"
    elif USE_EMBEDDED_FROZEN_ARTIFACTS:
        INPUT_PATHS["frozen_bundle"] = EMBEDDED_FROZEN_PATHS[FROZEN_BUNDLE_FILENAME]
        FROZEN_INPUT_SOURCE = "embedded checksum-verified fallback"
    else:
        raise FileNotFoundError(
            f"Could not find frozen bundle {FROZEN_BUNDLE_FILENAME!r}. "
            "Set FROZEN_BUNDLE_PATH or enable USE_EMBEDDED_FROZEN_ARTIFACTS."
        )

path_table = pd.DataFrame(
    [{"Input": key, "Resolved path": str(value)} for key, value in INPUT_PATHS.items()]
)
display(path_table)
print("Frozen final-artifact source:", FROZEN_INPUT_SOURCE if not separate_frozen_paths else "three explicit paths")
print("Output directory:", OUTPUT_DIR)
print("Intermediate artifact directory:", ARTIFACT_DIR)

In [ ]:
# STEP 2 — Load the frozen model, MinMaxScaler, and feature selector.
def _corr_abs(A, b, method):
    if method == "spearman":
        from scipy.stats import rankdata
        A = np.apply_along_axis(rankdata, 0, A)
        b = rankdata(b)
    centered_y = b - b.mean()
    centered_X = A - A.mean(0)
    numerator = (centered_X * centered_y[:, None]).sum(0)
    denominator = np.sqrt((centered_X**2).sum(0)) * np.sqrt((centered_y**2).sum())
    return np.abs(numerator / (denominator + 1e-12))

sys.modules["__main__"]._corr_abs = _corr_abs

def load_frozen_objects():
    if separate_frozen_paths:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model = joblib.load(INPUT_PATHS["frozen_model"])
            scaler = joblib.load(INPUT_PATHS["frozen_scaler"])
            selector = joblib.load(INPUT_PATHS["frozen_selector"])
        metadata = {
            "dataset": RUN_DATASET, "FS_Method": RUN_METHOD,
            "FS_k": EXPECTED_SELECTED_K, "model_name": EXPECTED_MODEL_CLASS,
            "TSR2": EXPECTED_REFERENCE_TEST_R2,
        }
        return model, scaler, selector, metadata

    bundle_path = INPUT_PATHS["frozen_bundle"]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        if bundle_path.suffix.lower() == ".json":
            descriptor = json.loads(bundle_path.read_text(encoding="utf-8"))
            if descriptor.get("format") != "frozen_component_bundle_v1":
                raise AssertionError(f"Unsupported frozen descriptor: {bundle_path}")
            components = descriptor["components"]
            model = joblib.load(bundle_path.parent / components["model"])
            scaler = joblib.load(bundle_path.parent / components["normalizer"])
            selector = joblib.load(bundle_path.parent / components["selector"])
            metadata = dict(descriptor["metadata"])
        else:
            payload = joblib.load(bundle_path)
            model = payload["model"]
            scaler = payload["normalizer"]
            selector = payload["selector"]
            metadata = {key: value for key, value in payload.items() if key not in {"model", "normalizer", "selector"}}
    return model, scaler, selector, metadata

frozen_model, frozen_scaler, frozen_selector, frozen_metadata = load_frozen_objects()
selected_indices = np.asarray(frozen_selector.get_support(indices=True), dtype=int)

checks = {
    "model class": frozen_model.__class__.__name__ == EXPECTED_MODEL_CLASS,
    "selected feature count": len(selected_indices) == EXPECTED_SELECTED_K,
    "scaler input dimension": int(frozen_scaler.n_features_in_) == 1024,
    "selector input dimension": int(frozen_selector.n_features_in_) == 1024,
    "model input dimension": int(frozen_model.n_features_in_) == EXPECTED_SELECTED_K,
}
failed = [name for name, passed in checks.items() if not passed]
if failed:
    raise AssertionError(f"Frozen artifact contract failed: {failed}")

display(pd.DataFrame([{"Check": key, "Passed": value} for key, value in checks.items()]))
print("Model:", frozen_model)
print("Scaler:", frozen_scaler.__class__.__name__)
print("Selector:", frozen_selector.__class__.__name__)
print("Selected HStack1024 indices:", selected_indices.tolist())

In [ ]:
# STEP 3 — Clean the raw receptor CSV and recreate the locked 60/20/20 split.
from rdkit import Chem, RDLogger
from sklearn.model_selection import train_test_split

RDLogger.DisableLog("rdApp.*")

def unordered_canonical_smiles_hash(values):
    """Hash canonical molecule membership without depending on row order."""
    ordered_values = sorted(str(value) for value in values)
    return hashlib.sha256("\n".join(ordered_values).encode("utf-8")).hexdigest()

def unordered_canonical_smiles_target_hash(smiles_values, target_values):
    """Hash canonical-SMILES/target pairs while ignoring only row order."""
    if len(smiles_values) != len(target_values):
        raise AssertionError("SMILES/target length mismatch")
    records = sorted(
        (str(smiles), np.float64(target).hex())
        for smiles, target in zip(smiles_values, target_values)
    )
    payload = "\n".join(f"{smiles}\t{target_hex}" for smiles, target_hex in records)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def canonical_split_hash(canonical_smiles):
    ordered_values = sorted(str(value) for value in canonical_smiles)
    return hashlib.md5("\n".join(ordered_values).encode("utf-8")).hexdigest()[:16]

raw_frame = pd.read_csv(INPUT_PATHS["raw_csv"])
required_columns = {"Smiles", "pIC50"}
missing_columns = required_columns.difference(raw_frame.columns)
if missing_columns:
    raise ValueError(f"Raw CSV is missing columns: {sorted(missing_columns)}")

valid_smiles_mask = raw_frame["Smiles"].apply(
    lambda value: Chem.MolFromSmiles(str(value)) is not None
)
clean_frame = raw_frame.loc[valid_smiles_mask].drop_duplicates(subset=["Smiles"]).copy()
clean_frame = clean_frame.loc[~np.isinf(clean_frame["pIC50"].to_numpy())].copy()
smiles = clean_frame["Smiles"].to_numpy().astype(str)
targets = clean_frame["pIC50"].to_numpy(dtype=float)
canonical_smiles = np.asarray([
    Chem.MolToSmiles(Chem.MolFromSmiles(value)) for value in smiles
]).astype(str)
canonical_by_raw_smiles = dict(zip(smiles.tolist(), canonical_smiles.tolist()))

bins = pd.qcut(targets, q=10, labels=False, duplicates="drop")
train_smiles, temp_smiles, train_y, temp_y, _, temp_bins = train_test_split(
    smiles, targets, bins, test_size=0.40, random_state=0, stratify=bins
)
val_smiles, test_smiles, val_y, test_y = train_test_split(
    temp_smiles, temp_y, test_size=0.50, random_state=0, stratify=temp_bins
)
splits = {
    "train": {"smiles": np.asarray(train_smiles).astype(str), "y": np.asarray(train_y, dtype=float)},
    "val": {"smiles": np.asarray(val_smiles).astype(str), "y": np.asarray(val_y, dtype=float)},
    "test": {"smiles": np.asarray(test_smiles).astype(str), "y": np.asarray(test_y, dtype=float)},
}
for payload in splits.values():
    payload["canonical_smiles"] = np.asarray([
        canonical_by_raw_smiles[value] for value in payload["smiles"]
    ]).astype(str)

if len(val_y) != EXPECTED_N_VAL or len(test_y) != EXPECTED_N_TEST:
    raise AssertionError(
        f"Split size changed: val={len(val_y)}, test={len(test_y)}; "
        f"expected val={EXPECTED_N_VAL}, test={EXPECTED_N_TEST}"
    )

# train_test_split can return the same stratified membership in a different row
# order across sklearn/NumPy versions. Validate canonical molecule membership
# and every canonical-SMILES/target pairing, but do not require a row order or
# one particular equivalent textual representation of a SMILES molecule.
split_smiles_sets = {
    split_name: set(payload["canonical_smiles"].tolist())
    for split_name, payload in splits.items()
}
for split_name, payload in splits.items():
    if len(split_smiles_sets[split_name]) != len(payload["canonical_smiles"]):
        raise AssertionError(f"{split_name} contains duplicate canonical molecules")
if split_smiles_sets["train"] & split_smiles_sets["val"]:
    raise AssertionError("Train/validation SMILES leakage detected")
if split_smiles_sets["train"] & split_smiles_sets["test"]:
    raise AssertionError("Train/test SMILES leakage detected")
if split_smiles_sets["val"] & split_smiles_sets["test"]:
    raise AssertionError("Validation/test SMILES leakage detected")
if set().union(*split_smiles_sets.values()) != set(canonical_smiles.tolist()):
    raise AssertionError("Split union does not reproduce the cleaned dataset")

for split_name, expected in EXPECTED_CANONICAL_HASHES.items():
    actual = canonical_split_hash(splits[split_name]["canonical_smiles"])
    if actual != expected:
        raise AssertionError(f"{split_name} canonical hash changed: {actual} != {expected}")

split_identity_audit = []
for split_name in ("train", "val", "test"):
    payload = splits[split_name]
    actual_smiles_hash = unordered_canonical_smiles_hash(payload["canonical_smiles"])
    actual_pair_hash = unordered_canonical_smiles_target_hash(
        payload["canonical_smiles"], payload["y"]
    )
    if actual_smiles_hash != EXPECTED_CANONICAL_SMILES_HASHES[split_name]:
        raise AssertionError(
            f"{split_name} canonical molecule membership changed: "
            f"{actual_smiles_hash} != {EXPECTED_CANONICAL_SMILES_HASHES[split_name]}"
        )
    if actual_pair_hash != EXPECTED_CANONICAL_PAIRED_HASHES[split_name]:
        raise AssertionError(
            f"{split_name} canonical-SMILES/target pairing changed: "
            f"{actual_pair_hash} != {EXPECTED_CANONICAL_PAIRED_HASHES[split_name]}"
        )
    split_identity_audit.append({
        "Split": split_name,
        "Rows": len(payload["y"]),
        "Unique_Canonical_Molecules": len(split_smiles_sets[split_name]),
        "Canonical_membership_hash": actual_smiles_hash,
        "Canonical_SMILES_target_pair_hash": actual_pair_hash,
    })

raw_split_path = ARTIFACT_DIR / "01_raw_splits.npz"
np.savez_compressed(
    raw_split_path,
    train_smiles=splits["train"]["smiles"], train_y=splits["train"]["y"],
    val_smiles=splits["val"]["smiles"], val_y=splits["val"]["y"],
    test_smiles=splits["test"]["smiles"], test_y=splits["test"]["y"],
    train_canonical_smiles=splits["train"]["canonical_smiles"],
    val_canonical_smiles=splits["val"]["canonical_smiles"],
    test_canonical_smiles=splits["test"]["canonical_smiles"],
)

print("Raw rows:", len(raw_frame))
print("Clean rows:", len(clean_frame))
print("Split sizes:", {name: len(payload["y"]) for name, payload in splits.items()})
print("Split identity verified without requiring version-specific row order.")
display(pd.DataFrame(split_identity_audit))
print("Saved:", raw_split_path)
display(pd.DataFrame({"test_smiles": test_smiles[:5], "test_pIC50": test_y[:5]}))

## Fresh 4-family feature extraction

The complete frozen inference implementation is embedded in the following cell so the model architectures, pooling, checkpoints, and alignment rules can be inspected directly.

In [ ]:
# STEP 4 — Complete frozen feature-extractor definitions.
# This source is embedded in the notebook for direct inspection; it is not imported from GitHub.
import warnings
import joblib


from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Dict, Optional, Sequence

import numpy as np

FAMILY_DIM = 256

def normalize_dataset(dataset):
    value = str(dataset).upper()
    return "ER" if value == "ERALPHA" else value

def checkpoint_path(dataset, family, repo_root=None):
    return INPUT_PATHS[f"{family}_checkpoint"]

def fingerprint_transformer_path(dataset, repo_root=None):
    return INPUT_PATHS["fingerprint_transformer"]

def load_joblib(path):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return joblib.load(path)

@dataclass(frozen=True)
class FeatureSplit:
    smiles: np.ndarray
    X: np.ndarray
    y: np.ndarray


MOLFORMER_REPO = "ibm/MoLFormer-XL-both-10pct"
MOLFORMER_REVISION = "7b12d946c181a37f6012b9dc3b002275de070314"
BIOT5_REPO = "QizhiPei/biot5-plus-base"
BIOT5_REVISION = "622b6a04bf0fe266c67dd0927374b00bf21eb7de"
MAX_LENGTH = 128
BATCH_SIZE = 32
INFERENCE_DEVICE = "cpu"


def _seed_cpu() -> None:
    import random
    import torch

    random.seed(0)
    np.random.seed(0)
    torch.manual_seed(0)
    torch.use_deterministic_algorithms(True, warn_only=True)


def _device():
    import torch

    # The research-replay contract is CPU-only. Never select CUDA implicitly:
    # different device kernels are outside the locked inference environment.
    return torch.device(INFERENCE_DEVICE)


def _require_256(name: str, features: np.ndarray, n_rows: int) -> np.ndarray:
    features = np.asarray(features, dtype=np.float32)
    if features.shape != (n_rows, FAMILY_DIM):
        raise AssertionError(
            f"{name}: expected {(n_rows, FAMILY_DIM)}, got {features.shape}"
        )
    if not np.isfinite(features).all():
        raise AssertionError(f"{name}: non-finite feature value")
    return features


def _transformer_dataset(sequences: Sequence[str], tokenizer):
    import torch
    from torch.utils.data import Dataset

    class SequenceDataset(Dataset):
        def __len__(self):
            return len(sequences)

        def __getitem__(self, index):
            encoded = tokenizer(
                str(sequences[index]),
                padding="max_length",
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
            )
            return {key: value.squeeze(0) for key, value in encoded.items()}

    return SequenceDataset()


def _molformer_regressor(state_dict, dataset: str):
    import torch.nn as nn

    first_dim = int(state_dict["regressor.0.weight"].shape[0])
    dropout = 0.25 if normalize_dataset(dataset) == "ER" else 0.2
    return nn.Sequential(
        nn.Linear(768, first_dim),
        nn.LayerNorm(first_dim),
        nn.GELU(),
        nn.Dropout(dropout),
        nn.Linear(first_dim, 256),
        nn.LayerNorm(256),
        nn.GELU(),
        nn.Dropout(dropout),
        nn.Linear(256, 1),
    )


def extract_smiles(
    dataset: str, smiles: Sequence[str], repo_root: Optional[Path] = None
) -> np.ndarray:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader
    from transformers import AutoConfig, AutoModel, AutoTokenizer

    _seed_cpu()
    device = _device()
    state = torch.load(
        checkpoint_path(dataset, "smiles", repo_root),
        map_location="cpu",
        weights_only=True,
    )
    hub_args = {
        "trust_remote_code": True,
        "revision": MOLFORMER_REVISION,
        # Some repository auto_map entries explicitly name the source repo.
        # Without code_revision, Transformers can load remote Python from the
        # current HEAD while loading weights/config.json from revision, creating
        # two distinct MolformerConfig classes and a registration failure.
        "code_revision": MOLFORMER_REVISION,
    }
    config = AutoConfig.from_pretrained(
        MOLFORMER_REPO, deterministic_eval=True, **hub_args
    )
    tokenizer = AutoTokenizer.from_pretrained(MOLFORMER_REPO, **hub_args)
    base = AutoModel.from_pretrained(MOLFORMER_REPO, config=config, **hub_args)

    class FrozenMolFormer(nn.Module):
        def __init__(self):
            super().__init__()
            self.transformer = base
            self.regressor = _molformer_regressor(state, dataset)

    model = FrozenMolFormer()
    model.load_state_dict(state)
    model.to(device).eval()
    flags = [module.deterministic for module in model.modules() if hasattr(module, "deterministic")]
    if not flags or not all(flags):
        raise AssertionError("MoLFormer deterministic_eval is not active")
    loader = DataLoader(_transformer_dataset(smiles, tokenizer), batch_size=BATCH_SIZE)
    chunks = []
    with torch.no_grad():
        for batch in loader:
            output = model.transformer(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
            )
            # Historical protocol: unmasked mean across all 128 token positions.
            pooled = output.last_hidden_state.mean(dim=1)
            chunks.append(model.regressor[:8](pooled).cpu().numpy())
    return _require_256("smiles", np.vstack(chunks), len(smiles))


def extract_selfies(
    dataset: str, smiles: Sequence[str], repo_root: Optional[Path] = None
) -> np.ndarray:
    import selfies as sf
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader
    from transformers import AutoConfig, AutoModel, AutoTokenizer

    _seed_cpu()
    device = _device()
    sequences = []
    for index, value in enumerate(smiles):
        try:
            sequences.append(sf.encoder(str(value)))
        except Exception as exc:
            raise ValueError(f"SELFIES conversion failed at row {index}: {value!r}") from exc
    state = torch.load(
        checkpoint_path(dataset, "selfies", repo_root),
        map_location="cpu",
        weights_only=True,
    )
    hub_args = {
        "trust_remote_code": True,
        "revision": BIOT5_REVISION,
        "code_revision": BIOT5_REVISION,
    }
    config = AutoConfig.from_pretrained(BIOT5_REPO, **hub_args)
    tokenizer = AutoTokenizer.from_pretrained(BIOT5_REPO, **hub_args)
    base = AutoModel.from_pretrained(BIOT5_REPO, config=config, **hub_args)

    class FrozenBioT5(nn.Module):
        def __init__(self):
            super().__init__()
            self.transformer = base
            hidden = int(base.config.hidden_size)
            self.regressor = nn.Sequential(
                nn.Linear(hidden, 512),
                nn.LayerNorm(512),
                nn.GELU(),
                nn.Linear(512, 256),
                nn.LayerNorm(256),
                nn.GELU(),
                nn.Linear(256, 1),
            )

    model = FrozenBioT5()
    model.load_state_dict(state)
    model.to(device).eval()
    loader = DataLoader(_transformer_dataset(sequences, tokenizer), batch_size=BATCH_SIZE)
    chunks = []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            decoder_ids = torch.zeros(
                (ids.size(0), 1), dtype=torch.long, device=device
            )
            output = model.transformer(
                input_ids=ids,
                attention_mask=mask,
                decoder_input_ids=decoder_ids,
            )
            pooled = output.last_hidden_state.mean(dim=1)
            chunks.append(model.regressor[:6](pooled).cpu().numpy())
    return _require_256("selfies", np.vstack(chunks), len(smiles))


def extract_fingerprint(
    dataset: str, smiles: Sequence[str], repo_root: Optional[Path] = None
) -> np.ndarray:
    import torch
    import torch.nn as nn

    _seed_cpu()
    device = _device()
    transformer = load_joblib(fingerprint_transformer_path(dataset, repo_root))
    ecfp = np.asarray(transformer.transform(list(smiles)), dtype=np.float32)
    if ecfp.shape != (len(smiles), 2048):
        raise AssertionError(f"fingerprint: expected {(len(smiles), 2048)}, got {ecfp.shape}")

    class FrozenECFPRegressor(nn.Module):
        def __init__(self):
            super().__init__()
            self.regressor = nn.Sequential(
                nn.Linear(2048, 1024),
                nn.LayerNorm(1024),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(1024, 256),
                nn.LayerNorm(256),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(256, 1),
            )

    model = FrozenECFPRegressor()
    model.load_state_dict(
        torch.load(
            checkpoint_path(dataset, "fingerprint", repo_root),
            map_location="cpu",
            weights_only=True,
        )
    )
    model.to(device).eval()
    with torch.no_grad():
        features = model.regressor[:8](torch.from_numpy(ecfp).to(device)).cpu().numpy()
    return _require_256("fingerprint", features, len(smiles))


def extract_graph(
    dataset: str, smiles: Sequence[str], repo_root: Optional[Path] = None
) -> np.ndarray:
    import deepchem as dc
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from rdkit import Chem
    from torch_geometric.data import Data
    from torch_geometric.loader import DataLoader
    from torch_geometric.nn import ChebConv, global_max_pool, global_mean_pool

    _seed_cpu()
    device = _device()
    featurizer = dc.feat.MolGraphConvFeaturizer(use_edges=True)
    graphs = []
    for index, value in enumerate(smiles):
        smi = str(value)
        try:
            if Chem.MolFromSmiles(smi) is None:
                raise ValueError("invalid RDKit SMILES")
            graph = featurizer.featurize([smi])[0]
            if graph.node_features is None or graph.num_nodes == 0:
                raise ValueError("empty graph")
            graphs.append(
                Data(
                    x=torch.tensor(graph.node_features, dtype=torch.float32),
                    edge_index=torch.tensor(graph.edge_index, dtype=torch.long),
                    edge_attr=torch.tensor(graph.edge_features, dtype=torch.float32),
                )
            )
        except Exception as exc:
            # The original notebooks silently skipped failed graphs.  That is not
            # safe for an aligned four-family pipeline, so fail without reordering.
            raise ValueError(f"Graph conversion failed at row {index}: {smi!r}") from exc

    class FrozenChebRegressorV3(nn.Module):
        def __init__(self):
            super().__init__()
            self.input_proj = nn.Linear(30, 256)
            self.convs = nn.ModuleList([ChebConv(256, 256, 3) for _ in range(3)])
            self.lns = nn.ModuleList([nn.LayerNorm(256) for _ in range(3)])
            self.fc = nn.Sequential(
                nn.Linear(512, 256), nn.LayerNorm(256), nn.GELU(), nn.Linear(256, 1)
            )

    model = FrozenChebRegressorV3()
    model.load_state_dict(
        torch.load(
            checkpoint_path(dataset, "graph", repo_root),
            map_location="cpu",
            weights_only=True,
        )
    )
    model.to(device).eval()
    chunks = []
    with torch.no_grad():
        for batch in DataLoader(graphs, batch_size=max(1, len(graphs)), shuffle=False):
            batch = batch.to(device)
            x = F.relu(model.input_proj(batch.x))
            for convolution, layer_norm in zip(model.convs, model.lns):
                x = x + F.relu(layer_norm(convolution(x, batch.edge_index)))
            mean = global_mean_pool(x, batch.batch)
            maximum = global_max_pool(x, batch.batch)
            chunks.append(
                model.fc[:3](torch.cat([mean, maximum], dim=-1)).cpu().numpy()
            )
    return _require_256("graph", np.vstack(chunks), len(smiles))


EXTRACTORS: Dict[str, Callable[..., np.ndarray]] = {
    "smiles": extract_smiles,
    "selfies": extract_selfies,
    "graph": extract_graph,
    "fingerprint": extract_fingerprint,
}


def extract_all(
    dataset: str,
    smiles: Sequence[str],
    targets: Sequence[float],
    repo_root: Optional[Path] = None,
) -> Dict[str, FeatureSplit]:
    ds = normalize_dataset(dataset)
    smi = np.asarray(smiles).astype(str)
    y = np.asarray(targets)
    return {
        family: FeatureSplit(smi, EXTRACTORS[family](ds, smi, repo_root), y)
        for family in ("smiles", "selfies", "graph", "fingerprint")
    }

In [ ]:
# STEP 5 — Prepare one aligned Train+Validation+Test sequence for fresh extraction.
SPLIT_ORDER = ("train", "val", "test")
split_bounds = {}
all_smiles_parts = []
all_y_parts = []
start = 0
for split_name in SPLIT_ORDER:
    stop = start + len(splits[split_name]["y"])
    split_bounds[split_name] = (start, stop)
    all_smiles_parts.append(splits[split_name]["smiles"])
    all_y_parts.append(splits[split_name]["y"])
    start = stop

all_smiles = np.concatenate(all_smiles_parts).astype(str)
all_y = np.concatenate(all_y_parts).astype(float)
family_features = {}

def split_feature_matrix(matrix):
    return {
        split_name: np.asarray(matrix[left:right], dtype=np.float32)
        for split_name, (left, right) in split_bounds.items()
    }

def save_family_features(family):
    output_path = ARTIFACT_DIR / f"02_{family}_features_256.npz"
    np.savez_compressed(output_path, **family_features[family])
    print(f"{family}: shapes =", {key: value.shape for key, value in family_features[family].items()})
    print(f"{family}: test[0, :5] =", family_features[family]["test"][0, :5])
    print("Saved:", output_path)

print("Combined extraction rows:", len(all_smiles))
print("Boundaries:", split_bounds)

In [ ]:
# STEP 6A — Extract fresh SMILES features (256 dimensions) and save them.
smiles_all = extract_smiles(RUN_DATASET, all_smiles)
family_features['smiles'] = split_feature_matrix(smiles_all)
save_family_features('smiles')

In [ ]:
# STEP 6B — Extract fresh SELFIES features (256 dimensions) and save them.
selfies_all = extract_selfies(RUN_DATASET, all_smiles)
family_features['selfies'] = split_feature_matrix(selfies_all)
save_family_features('selfies')

In [ ]:
# STEP 6C — Extract fresh GRAPH features (256 dimensions) and save them.
graph_all = extract_graph(RUN_DATASET, all_smiles)
family_features['graph'] = split_feature_matrix(graph_all)
save_family_features('graph')

In [ ]:
# STEP 6D — Extract fresh FINGERPRINT features (256 dimensions) and save them.
fingerprint_all = extract_fingerprint(RUN_DATASET, all_smiles)
family_features['fingerprint'] = split_feature_matrix(fingerprint_all)
save_family_features('fingerprint')

In [ ]:
# STEP 7 — Concatenate four 256-dimensional families into HStack1024 and save it.
FAMILY_ORDER = ("smiles", "selfies", "graph", "fingerprint")
hstack = {}
for split_name in SPLIT_ORDER:
    matrices = [family_features[family][split_name] for family in FAMILY_ORDER]
    row_counts = {matrix.shape[0] for matrix in matrices}
    if row_counts != {len(splits[split_name]["y"])}:
        raise AssertionError(f"{split_name}: family row alignment failed")
    hstack[split_name] = np.hstack(matrices).astype(np.float32)
    expected_shape = (len(splits[split_name]["y"]), 1024)
    if hstack[split_name].shape != expected_shape:
        raise AssertionError(f"{split_name}: expected {expected_shape}, got {hstack[split_name].shape}")

hstack_path = ARTIFACT_DIR / "03_hstack1024.npz"
np.savez_compressed(hstack_path, **hstack)
print("HStack shapes:", {key: value.shape for key, value in hstack.items()})
print("Test HStack first five values:", hstack["test"][0, :5])
print("Saved:", hstack_path)

In [ ]:
# STEP 8 — Apply the frozen MinMaxScaler without fitting and save scaled HStack1024.
def apply_frozen_minmax(X, scaler):
    transformed = np.asarray(X) * np.asarray(scaler.scale_) + np.asarray(scaler.min_)
    return transformed.astype(np.float32)

scaled = {
    split_name: apply_frozen_minmax(hstack[split_name], frozen_scaler)
    for split_name in SPLIT_ORDER
}
scaled_path = ARTIFACT_DIR / "04_minmax_scaled_hstack1024.npz"
np.savez_compressed(scaled_path, **scaled)
print("Scaled shapes:", {key: value.shape for key, value in scaled.items()})
print("Test scaled range:", float(scaled["test"].min()), "to", float(scaled["test"].max()))
print("Test scaled first five values:", scaled["test"][0, :5])
print("Saved:", scaled_path)

In [ ]:
# STEP 9 — Apply the frozen feature selector without fitting and save selected matrices.
selected = {
    split_name: np.asarray(scaled[split_name])[:, selected_indices].astype(np.float32)
    for split_name in SPLIT_ORDER
}
for split_name, matrix in selected.items():
    expected_shape = (len(splits[split_name]["y"]), EXPECTED_SELECTED_K)
    if matrix.shape != expected_shape:
        raise AssertionError(f"{split_name}: expected selected shape {expected_shape}, got {matrix.shape}")

selected_path = ARTIFACT_DIR / f"05_selected_features_k{EXPECTED_SELECTED_K}.npz"
np.savez_compressed(
    selected_path,
    selected_indices=selected_indices,
    train=selected["train"], val=selected["val"], test=selected["test"],
)
print("Selected HStack1024 indices:", selected_indices.tolist())
print("Selected shapes:", {key: value.shape for key, value in selected.items()})
print("Test selected first five values:", selected["test"][0, :5])
print("Saved:", selected_path)

In [ ]:
# STEP 10 — Predict the locked Test split with the frozen final regressor and save results.
def predict_frozen_linear(X, model):
    features = np.asarray(X, dtype=np.float64)
    coefficients = np.asarray(model.coef_, dtype=np.float64).reshape(1, -1)
    intercept = np.asarray(model.intercept_, dtype=np.float64)
    return (np.sum(features * coefficients, axis=1) + intercept).reshape(-1)

y_pred = predict_frozen_linear(selected["test"], frozen_model)
test_predictions = pd.DataFrame(
    {
        "Test_Row": np.arange(len(test_y), dtype=int),
        "Smiles": test_smiles,
        "y_true_pIC50": test_y,
        "y_pred_pIC50": y_pred,
        "Residual_Pred_minus_Obs": y_pred - test_y,
    }
)
predictions_path = ARTIFACT_DIR / "06_test_predictions.csv"
test_predictions.to_csv(predictions_path, index=False)
print("Prediction count:", len(y_pred))
print("Prediction range:", float(y_pred.min()), "to", float(y_pred.max()))
print("Saved:", predictions_path)
display(test_predictions.head())

In [ ]:
# STEP 11 — Regression metrics and transparent kNN applicability-domain functions.
from scipy.stats import pearsonr, spearmanr
from sklearn.neighbors import NearestNeighbors

def regression_metrics(y_true, y_prediction):
    observed = np.asarray(y_true, dtype=np.float64).reshape(-1)
    predicted = np.asarray(y_prediction, dtype=np.float64).reshape(-1)
    residual = predicted - observed
    ss_res = float(np.sum(residual * residual))
    centered = observed - float(np.mean(observed))
    ss_tot = float(np.sum(centered * centered))
    r2 = float("nan") if len(observed) < 2 or ss_tot == 0.0 else 1.0 - ss_res / ss_tot
    pearson = (
        float("nan") if len(observed) < 2 or np.std(observed) == 0.0 or np.std(predicted) == 0.0
        else float(pearsonr(observed, predicted)[0])
    )
    spearman = (
        float("nan") if len(observed) < 2 or np.std(observed) == 0.0 or np.std(predicted) == 0.0
        else float(spearmanr(observed, predicted)[0])
    )
    return {
        "R2": r2,
        "RMSE": float(np.sqrt(np.mean(residual * residual))),
        "MAE": float(np.mean(np.abs(residual))),
        "ME": float(np.mean(residual)),
        "Pearson": pearson,
        "Spearman": spearman,
    }

def threshold_token(multiplier):
    text = np.format_float_positional(float(multiplier), trim="-")
    if "." not in text:
        text += ".0"
    return text.replace("-", "m").replace(".", "p")

def summary_row(label, multiplier, threshold, mask):
    metrics = regression_metrics(test_y[mask], y_pred[mask])
    n_ind = int(mask.sum())
    return {
        "K": label,
        "Threshold_Multiplier": float(multiplier),
        "AD_Distance_Threshold": float(threshold),
        **metrics,
        "N_Test": int(len(test_y)),
        "INDs": n_ind,
        "Coverage": float(n_ind / len(test_y)),
        "OODs": int(len(test_y) - n_ind),
    }

print("AD functions ready")

In [ ]:
# STEP 12 — Build Train+Validation AD reference, evaluate No AD and k=3...25, save all outputs.
X_reference = np.vstack([selected["train"], selected["val"]]).astype(np.float32)
X_test_ad = selected["test"].astype(np.float32)
reference_path = ARTIFACT_DIR / "07_ad_reference_train_plus_validation.npy"
np.save(reference_path, X_reference)

no_ad_metrics = regression_metrics(test_y, y_pred)
if abs(float(no_ad_metrics["R2"]) - EXPECTED_REFERENCE_TEST_R2) > R2_TOLERANCE:
    raise AssertionError(
        f"No-AD Test R2={no_ad_metrics['R2']:.12f}, "
        f"reference={EXPECTED_REFERENCE_TEST_R2:.12f}, tolerance={R2_TOLERANCE}"
    )

rows_by_multiplier = {float(value): [] for value in AD_THRESHOLD_MULTIPLIERS}
details_by_multiplier = {
    float(value): test_predictions.copy() for value in AD_THRESHOLD_MULTIPLIERS
}
all_test_mask = np.ones(len(test_y), dtype=bool)
for multiplier in rows_by_multiplier:
    rows_by_multiplier[multiplier].append(
        summary_row("No AD", multiplier, float("nan"), all_test_mask)
    )

for ad_k in AD_NEIGHBOR_K_VALUES:
    knn = NearestNeighbors(n_neighbors=ad_k, metric="euclidean", algorithm="auto")
    knn.fit(X_reference)
    reference_distances, _ = knn.kneighbors(X_reference)
    test_distances, _ = knn.kneighbors(X_test_ad)
    reference_mean_distance = reference_distances.mean(axis=1)
    test_mean_distance = test_distances.mean(axis=1)
    center = float(reference_mean_distance.mean())
    spread = float(reference_mean_distance.std(ddof=0))
    for multiplier in sorted(rows_by_multiplier):
        threshold = center + multiplier * spread
        ind_mask = np.asarray(test_mean_distance <= threshold, dtype=bool)
        if not ind_mask.any():
            raise ValueError(f"No IND Test rows remain for k={ad_k}, multiplier={multiplier}")
        rows_by_multiplier[multiplier].append(
            summary_row(f"k={ad_k}", multiplier, threshold, ind_mask)
        )
        detail = details_by_multiplier[multiplier]
        detail[f"k{ad_k}_Mean_Distance"] = test_mean_distance
        detail[f"k{ad_k}_Threshold"] = threshold
        detail[f"k{ad_k}_IND"] = ind_mask

summary_paths = {}
detail_paths = {}
for multiplier, rows in rows_by_multiplier.items():
    token = threshold_token(multiplier)
    summary = pd.DataFrame(rows)
    summary_path = OUTPUT_DIR / (
        f"AD_Test_Regression_{RUN_DATASET}_{RUN_METHOD}_threshold_{token}.csv"
    )
    detail_path = ARTIFACT_DIR / f"08_test_ad_pointwise_threshold_{token}.csv"
    summary.to_csv(summary_path, index=False)
    details_by_multiplier[multiplier].to_csv(detail_path, index=False)
    summary_paths[multiplier] = summary_path
    detail_paths[multiplier] = detail_path
    print(f"\nThreshold multiplier={multiplier}")
    print("Summary saved:", summary_path)
    print("Pointwise AD saved:", detail_path)
    display(summary)

print("AD reference shape:", X_reference.shape)
print("Test AD query shape:", X_test_ad.shape)
print("AD reference saved:", reference_path)

In [ ]:
# STEP 13 — Write reproducibility metadata and an SHA-256 artifact manifest.
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

metadata = {
    "dataset": RUN_DATASET,
    "method": RUN_METHOD,
    "raw_csv": str(INPUT_PATHS["raw_csv"]),
    "input_paths": {key: str(value) for key, value in INPUT_PATHS.items()},
    "split_rows": {key: int(len(value["y"])) for key, value in splits.items()},
    "feature_family_order": list(FAMILY_ORDER),
    "family_dimension": 256,
    "hstack_dimension": 1024,
    "selected_feature_count": int(EXPECTED_SELECTED_K),
    "selected_indices": selected_indices.tolist(),
    "model_class": frozen_model.__class__.__name__,
    "ad_reference": "Train + Validation selected features",
    "ad_query": "locked Test selected features",
    "ad_k_values": list(AD_NEIGHBOR_K_VALUES),
    "threshold_multipliers": list(AD_THRESHOLD_MULTIPLIERS),
    "threshold_formula": "mean(reference_mean_distance) + multiplier * std(reference_mean_distance, ddof=0)",
    "no_ad_metrics": no_ad_metrics,
}
metadata_path = OUTPUT_DIR / "run_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

artifact_paths = sorted(
    [path for path in OUTPUT_DIR.rglob("*") if path.is_file() and path.name != "artifact_manifest.csv"]
)
manifest_frame = pd.DataFrame(
    [
        {
            "Relative_Path": str(path.relative_to(OUTPUT_DIR)),
            "Bytes": path.stat().st_size,
            "SHA256": sha256_file(path),
        }
        for path in artifact_paths
    ]
)
manifest_path = OUTPUT_DIR / "artifact_manifest.csv"
manifest_frame.to_csv(manifest_path, index=False)

print("Metadata saved:", metadata_path)
print("Artifact manifest saved:", manifest_path)
print("Total saved files:", len(manifest_frame) + 1)
display(manifest_frame)

## Saved results

The two original threshold summary CSVs remain at the variant output root. Reloadable split, feature, HStack, scaled, selected, prediction, AD-reference, and pointwise IND/OOD artifacts are under `intermediate_artifacts/`. `artifact_manifest.csv` records every saved file and SHA-256 digest.